# 把 samples 上傳到 Google 雲端硬碟（在 JetBot 上跑）

用 **rclone**（單一執行檔，不依賴 Python 版本）把 `samples/` 裡的照片傳到 Google 雲端硬碟。

| 步驟 | 什麼時候跑 |
|---|---|
| ① 準備 rclone | **每次 Kernel 重啟後都要先跑**（已下載過不會重抓） |
| ② Google 授權 | 只做一次（或刪除授權後要重做） |
| ③ 上傳 | 每次要傳照片時 |
| ④ 刪除授權 | 傳完建議執行 |

**前提：JetBot 要能上網**（例如連手機熱點）。

這支 notebook 要放在 `samples/` 的**同一層資料夾**（`marker_calibration/`）。

程式的輸出一律是英文：容器的 locale 不是 UTF-8，`print()` 中文會出錯。

## ① 準備 rclone

第一次會下載約 20 MB、解壓後約 60 MB，裝在 `/workspace/bin`（SD 卡上，容器重開不會消失）。

In [ ]:
import os
import subprocess
import urllib.request
import zipfile

SRC = 'samples'                   # 要上傳的本機資料夾
DEST = 'jetbot_marker_samples'    # Google 雲端硬碟上的資料夾名稱
REMOTE = 'gdrive'                 # rclone 設定裡的名稱,不用改

BIN = '/workspace/bin'
RCLONE = BIN + '/rclone'
os.environ['RCLONE_CONFIG'] = BIN + '/rclone.conf'   # 設定檔也放 SD 卡,容器重開還在
if not os.path.isdir(BIN):
    os.makedirs(BIN)

# 手動備案:在筆電下載 rclone-...-linux-arm64.zip,用 Jupyter 上傳到這個資料夾,
# 檔名改成 rclone.zip,再跑這格。
MANUAL_ZIP = 'rclone.zip'

# urllib 預設的 User-Agent 是 "Python-urllib/3.6",部分下載站會直接回 403,
# 所以每個請求都帶一般瀏覽器的 User-Agent。
UA = {'User-Agent': 'Mozilla/5.0 (X11; Linux aarch64) rclone-installer'}


def fetch(url, dest):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=60) as r, open(dest, 'wb') as f:
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)


def github_asset_url():
    import json
    req = urllib.request.Request(
        'https://api.github.com/repos/rclone/rclone/releases/latest', headers=UA)
    with urllib.request.urlopen(req, timeout=30) as r:
        rel = json.loads(r.read().decode('utf-8'))
    for a in rel['assets']:
        if a['name'].endswith('-linux-arm64.zip'):
            return a['browser_download_url']
    raise RuntimeError('no linux-arm64 zip in latest release')


def install_from_zip(zpath):
    with zipfile.ZipFile(zpath) as z:
        member = [n for n in z.namelist() if n.endswith('/rclone')][0]
        with z.open(member) as src, open(RCLONE, 'wb') as dst:
            dst.write(src.read())
    os.chmod(RCLONE, 0o755)


if os.path.exists(RCLONE):
    print('rclone already installed:', RCLONE)
elif os.path.exists(MANUAL_ZIP):
    print('Installing from uploaded', MANUAL_ZIP)
    install_from_zip(MANUAL_ZIP)
    os.remove(MANUAL_ZIP)
    print('Installed to', RCLONE)
else:
    sources = [
        ('rclone.org', lambda: 'https://downloads.rclone.org/rclone-current-linux-arm64.zip'),
        ('GitHub', github_asset_url),
    ]
    for name, get_url in sources:
        try:
            url = get_url()
            print('Downloading from %s (about 20 MB) ...' % name)
            fetch(url, '/tmp/rclone.zip')
            install_from_zip('/tmp/rclone.zip')
            os.remove('/tmp/rclone.zip')
            print('Installed to', RCLONE)
            break
        except Exception as e:
            print('  %s failed: %s' % (name, e))
    else:
        raise SystemExit(
            'All downloads failed. Manual fallback:\n'
            '  1. On the laptop, download rclone-vX.XX.X-linux-arm64.zip from\n'
            '     https://github.com/rclone/rclone/releases/latest\n'
            '  2. Upload it to this folder with Jupyter, rename it to rclone.zip\n'
            '  3. Run this cell again')


def run(args):
    """Run rclone and return stdout (Python 3.6: no capture_output)."""
    p = subprocess.run([RCLONE] + args, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, universal_newlines=True)
    return p.returncode, p.stdout


def local_pngs():
    if not os.path.isdir(SRC):
        return []
    return sorted(n for n in os.listdir(SRC) if n.lower().endswith('.png'))


def is_authorized():
    _rc, out = run(['listremotes'])
    return (REMOTE + ':') in out.split()


print(run(['version'])[1].splitlines()[0])
print('Local photos in %s/: %d' % (SRC, len(local_pngs())))
print('Google Drive authorized:', is_authorized())

## ② Google 授權（只做一次）

JetBot 沒有瀏覽器，所以在**筆電**上登入 Google，再把授權碼貼過來。

**筆電（Windows PowerShell）**，第一次先安裝 rclone，裝完**關掉 PowerShell 重開**：

```powershell
winget install Rclone.Rclone
```

然後執行下面這行，瀏覽器會開 Google 登入頁，登入並允許：

```powershell
rclone authorize "drive"
```

終端機會印出一段 `{"access_token":...}`，**從 `{` 複製到 `}`**，執行下一格後貼進輸入框按 Enter。

輸入框用 `getpass`，授權碼**不會存進這個 notebook 檔**，推到 GitHub 也不會外洩。

In [ ]:
import getpass

if is_authorized():
    print('Already authorized. Skip this cell (or run step 4 first to re-authorize).')
else:
    token = getpass.getpass('Paste token JSON, then press Enter: ').strip()
    rc, out = run(['config', 'create', REMOTE, 'drive', 'token', token, '--non-interactive'])
    del token
    if rc != 0:
        print('FAILED to save config:')
        print(out)
    else:
        rc, out = run(['lsd', REMOTE + ':', '--max-depth', '1'])
        if rc == 0:
            print('OK. Top-level folders in your Google Drive:')
            print(out)
        else:
            print('Config saved but Google Drive is not reachable:')
            print(out)
            print('Check internet, or the token may be incomplete (copy from { to }).')

## ③ 上傳

已經傳過的檔案會自動跳過，可以重複執行。最後會比對本機與雲端的張數。

In [ ]:
names = local_pngs()
if not names:
    print('No .png in %s/ . Take photos with capture_samples.ipynb first.' % SRC)
elif not is_authorized():
    print('Not authorized yet. Run step 2 first.')
else:
    print('Uploading %d photos to %s:%s ...' % (len(names), REMOTE, DEST))
    # 用 get_ipython().system 而不是 !,才能在 if 區塊裡即時顯示進度
    get_ipython().system('%s copy %s %s:%s --include "*.png" --stats-one-line --stats 5s -v'
                         % (RCLONE, SRC, REMOTE, DEST))

    rc, out = run(['lsf', REMOTE + ':' + DEST, '--include', '*.png'])
    remote = set(out.split())
    missing = [n for n in names if n not in remote]
    print('')
    print('Local: %d   On Google Drive: %d' % (len(names), len(remote)))
    if missing:
        print('MISSING on Drive (run this cell again):')
        for n in missing:
            print('  ', n)
    else:
        print('All photos uploaded. Folder on Google Drive: %s' % DEST)

## ④ 刪除授權（傳完建議執行）

這組授權能存取你**整個** Google 雲端硬碟，而且存在 JetBot 上，組員也碰得到這台車。
傳完就刪，下次要用再做一次步驟 ②。照片留在雲端硬碟，不受影響。

In [ ]:
if is_authorized():
    rc, out = run(['config', 'delete', REMOTE])
    print('Authorization deleted.' if rc == 0 else out)
else:
    print('No authorization stored. Nothing to delete.')